In [18]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import joblib
import time
import ta
from datetime import datetime, timedelta, time as dt_time
import logging
import matplotlib.pyplot as plt
import optuna
import pickle
import os
import sys
from concurrent.futures import ThreadPoolExecutor, TimeoutError
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin

In [19]:
class TypeCaster(BaseEstimator, TransformerMixin):
    """Cast specified columns back to int32 after the ColumnTransformer."""
    def __init__(self, cat_idx):
        self.cat_idx = cat_idx
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy().astype(object)
        for idx in self.cat_idx:
            col = X[:, idx].astype(np.int32)
            X[:, idx] = col
        return X

In [20]:
def _rolling_r2(y: pd.Series, window: int):
    idx = np.arange(window)
    def _r2(a):
        y_ = a.astype(float); x_ = idx
        xm, ym = x_.mean(), y_.mean()
        cov = ((x_-xm)*(y_-ym)).sum(); var = ((x_-xm)**2).sum()
        if var == 0: return np.nan
        slope = cov/var; yhat = slope*(x_-xm)+ym
        ss_tot = ((y_-ym)**2).sum(); ss_res = ((y_-yhat)**2).sum()
        return 1 - (ss_res/ss_tot) if ss_tot>0 else np.nan
    return y.rolling(window).apply(_r2, raw=True)

def _donch_pos(c: pd.Series, h: pd.Series, l: pd.Series, window: int):
    hh = h.rolling(window).max(); ll = l.rolling(window).min()
    return ((c - ll) / (hh - ll)).clip(0, 1)

def _pct_above(series: pd.Series, ref: pd.Series, window: int):
    return (series > ref).rolling(window).mean()

In [1]:
# ==================== CONFIGURATION (v8 LIVE) ====================
class StrategyConfig:
    RISK_PER_TRADE: float    = 0.0015
    MAX_TRADES: int          = 7
    PARTIAL_CLOSE_PCT: float = 0.7
    MAX_DD: float            = 0.03

    # fallback if an instrument has no explicit threshold
    DEFAULT_THRESHOLD: float = 0.85

    # Instruments with 1H, 4H and 30M timeframes
    INSTRUMENTS = {
        # 1H models




        
        'GBPUSD.4H': {
            'pip':         0.0001,
            'pip_value':   10,
            'model_path':  'gbpusd4h_final_wfv65.pkl',
            'scaler_path': 'gbpusd_4h_v11_scaler.pkl',
            'threshold':   0.75,
        },

        'EURUSD.4H': {
            'pip':         0.0001,
            'pip_value':   10,
            'model_path':  'eurusd4h_final_wfv54.pkl',
            'scaler_path': 'eurusd_4h_v52_scaler.pkl',
            'threshold':   0.75,
        },



                

        
         'GBPJPY.4H': {
            'pip':         0.01,
            'pip_value':   6,
            'model_path':  'gbpjpy4h_final_wfv65.pkl',
            'scaler_path': 'gbpjpy_4h_v15_scaler.pkl',
            'threshold':   0.72,
        },




        'USDJPY.4H': {
            'pip':         0.01,
            'pip_value':   6.8,
            'model_path':  'usdjpy4h_final_wfv100.pkl',
            'scaler_path': 'usdjpy4H_v100_scaler.pkl',
            'threshold':   0.75,
        },


        'USDCHF.4H': {
            'pip':         0.0001,
            'pip_value':   12.7,
            'model_path':  'usdchf4h_final_wfv2.pkl',
            'scaler_path': 'usdchf_4h_v2_scaler.pkl',
            'threshold':   0.75,
        },






         'CADJPY.4H': {
            'pip':         0.01,
            'pip_value':   6.2,
            'model_path':  'cadjpy4h_final_wfv2.pkl',
            'scaler_path': 'cadjpy_4h_v2_scaler.pkl',
            'threshold':   0.75,
        },







    
    }


In [4]:
# ==================== STATE PERSISTENCE ====================
STATE_FILENAME = "open_positions.pkl"

def load_open_positions():
    if os.path.exists(STATE_FILENAME):
        try:
            with open(STATE_FILENAME, "rb") as f:
                return pickle.load(f)
        except Exception as e:
            print(f"Error loading state: {e}")
            return []
    return []

def save_open_positions(open_positions):
    try:
        with open(STATE_FILENAME, "wb") as f:
            pickle.dump(open_positions, f)
    except Exception as e:
        print(f"Error saving state: {e}")

In [23]:
# ==================== MT5 INITIALIZATION ====================
def initialize_mt5():
    mt5_path = "C:\\Program Files\\MetaTrader 5\\terminal64.exe"  # Adjust path if needed
    account =
    password = 
    server = "FTMO-Server5"
    if not mt5.initialize(path=mt5_path, timeout=30000):
        raise ConnectionError(f"MT5 init failed: {mt5.last_error()}")
    if not mt5.login(account, password, server):
        mt5.shutdown()
        raise ConnectionError("MT5 login failed")
    print(f"✅ Successfully connected to MT5 account: {account}")

# Define MT5 login credentials
account = 
password = 
server = "FTMO-Server5"
mt5_path = "C:\\Program Files\\MetaTrader 5\\terminal64.exe"  # Update if needed

if not mt5.initialize(path=mt5_path, timeout=30000):
    print("MT5 initialization failed, error:", mt5.last_error())
    quit()

if not mt5.login(account, password, server):
    print("Login failed, error code:", mt5.last_error())
    mt5.shutdown()
    quit()

print(f"✅ Successfully connected to MT5 account: {account}")


✅ Successfully connected to MT5 account: 550111923


In [24]:
# ==================== NEW CONNECTION HELPERS ====================
def kill_mt5_process():
    """Windows-specific cleanup"""
    if os.name == 'nt':
        os.system('taskkill /f /im terminal64.exe > nul 2>&1')

def reconnect_mt5(max_retries=3):
    """Auto-reconnect using YOUR original initialize_mt5()"""
    for attempt in range(max_retries):
        try:
            kill_mt5_process()
            initialize_mt5()  # Calls your existing function
            return True
        except ConnectionError as e:
            print(f"Reconnect attempt {attempt+1} failed: {str(e)}")
            time.sleep(15)
    return False

In [25]:
# ------------------- SYMBOL AVAILABILITY CHECK -------------------
def verify_symbol(symbol: str) -> bool:
    info = mt5.symbol_info(symbol)
    if not info or not info.visible:
        print(f"❌ Symbol {symbol} not available. MT5 Error: {mt5.last_error()}")
        return False
    return True

# ==================== DATA EXTRACTION & FEATURE ENGINEERING ====================
def get_timeframe_data(symbol: str, timeframe: str) -> pd.DataFrame:
    """
    Fetches and formats OHLC data for the given symbol and timeframe.
    Supports '30M', '1H', '4H', '1D', 'M1'.
    """
    try:
        tf_map = {
            'M1':  mt5.TIMEFRAME_M1,
            '30M': mt5.TIMEFRAME_M30,
            '1H':  mt5.TIMEFRAME_H1,
            '4H':  mt5.TIMEFRAME_H4,
            '1D':  mt5.TIMEFRAME_D1,
        }
        mt5_tf = tf_map.get(timeframe)
        if mt5_tf is None:
            raise ValueError(f"Unsupported timeframe: {timeframe}")

        rates = mt5.copy_rates_from_pos(symbol, mt5_tf, 0, 501)
        if rates is None or len(rates) < 100:
            print(f"⚠️ Insufficient data ({len(rates) if rates is not None else 0} bars) for {symbol} {timeframe}")
            return None

        df = pd.DataFrame(rates)
        df['time'] = pd.to_datetime(df['time'], unit='s')
        df.set_index('time', inplace=True)
        df.sort_index(ascending=True, inplace=True)
        df.rename(columns={'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close'}, inplace=True)
        df = df.iloc[:-1]
        return df

    except Exception as e:
        print(f"🚨 Critical error fetching data for {symbol} {timeframe}: {e}")
        return None

def calculate_features(df: pd.DataFrame, timeframe: str = '1H') -> pd.Series:
    # Expected columns: ATR_median100 is computed for risk but not included in model features.
    expected_cols = [
        'ema_diff', 'ema50_diff', 'bb_penetration', 'bb_lower_penetration', 'bb_width',
        'atr_ratio', 'adx', 'macd_hist',
        'ema200_slope', 'vol_ratio', 'trend_purity', 'EMA50_slope',
        'vol_regime','rsi14','momentum_flag','volume_ratio','trend_volatility','vwap','vwap_distance','ema_triple_diff','CMO14',
        'pct_above_ema200_100','r2_72','ema50_200_atr','donch_pos_100','EMA200_trend','trend_alignment'
    ]
    if df is None or df.empty or len(df) < 100:
        return pd.Series(index=expected_cols)
    
    df['EMA20'] = ta.trend.ema_indicator(df['Close'], 20)
    df['EMA50'] = ta.trend.ema_indicator(df['Close'], 50)
    df['EMA200'] = ta.trend.ema_indicator(df['Close'], 200)
    df['ema_diff'] = df['EMA20'] - df['EMA200']
    df['ema50_diff'] = df['EMA20'] - df['EMA50']
    
    df['EMA50_slope'] = df['EMA50'].diff(5) / df['EMA50'].shift(5)
    
    # Bollinger Bands + penetrations + width
    df['BB_upper'] = ta.volatility.bollinger_hband(df['Close'], 20, 2)
    df['bb_penetration'] = (df['Close'] - df['BB_upper']) / df['BB_upper']
    df['BB_lower'] = ta.volatility.bollinger_lband(df['Close'], 20, 2)
    df['bb_lower_penetration'] = (df['Close'] - df['BB_lower']) / df['BB_lower']
    df['BB_width'] = (df['BB_upper'] - df['BB_lower']) / df['Close']
    
    df['ATR'] = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'], 14)
    df['ATR_MA'] = df['ATR'].rolling(20).mean()
    
    df['atr_ratio'] = df['ATR'] / df['ATR_MA']
    
    df['ADX'] = ta.trend.ADXIndicator(df['High'], df['Low'], df['Close'], window=14).adx()
    
    macd = ta.trend.MACD(df['Close'])
    df['macd_hist'] = macd.macd_diff()
    
    df['ema200_slope'] = df['EMA200'].diff(10) / df['EMA200'].shift(10)
    
    df['STD_20'] = df['Close'].rolling(20).std()
    df['vol_ratio'] = df['ATR'] / df['STD_20'].where(df['STD_20'] > 1e-5, np.nan)
    
    df['trend_purity'] = df['Close'].rolling(20).corr(df['EMA20'].shift())
    
 
    
    df['vol_regime'] = np.where(
        df['ATR'] > df['ATR'].rolling(100).quantile(0.75),
        2,
        np.where(df['ATR'] > df['ATR'].rolling(100).quantile(0.4), 1, 0)
    )
    df['RSI14'] = ta.momentum.rsi(df['Close'], window=14)
    df['momentum_flag'] = np.sign(df['Close'].diff(5)) * ((abs(df['Close'].diff(5)) / df['ATR']) > 1).astype(int)

    df['ema_triple_diff'] = (df['EMA20'] - df['EMA50']) - (df['EMA50'] - df['EMA200'])

    if 'Volume' not in df.columns:
        if 'tick_volume' in df.columns:
            df['Volume'] = df['tick_volume']
        elif 'real_volume' in df.columns:
            df['Volume'] = df['real_volume']
        else:
            raise KeyError("No volume field found in DataFrame to compute volume_ratio")
   


    
    df['volume_MA20'] = df['Volume'].rolling(20).mean()
    df['volume_ratio'] = df['Volume'] / df['volume_MA20']
    df['trend_volatility'] = df['trend_purity'].rolling(10).std()

    df['vwap'] = ta.volume.volume_weighted_average_price(df['High'], df['Low'], df['Close'], df['Volume'], window=20)
    df['vwap_distance'] = (df['Close'] - df['vwap']) / df['vwap']
    mom = df['Close'].diff()
    pos = mom.clip(lower=0)
    neg = -mom.clip(upper=0)
    sum_pos = pos.rolling(14).sum()
    sum_neg = neg.rolling(14).sum()
    df['CMO14'] = (sum_pos - sum_neg) / (sum_pos + sum_neg)

    df['EMA200_trend'] = (df['Close'] > df['EMA200']).astype(int)
    df['EMA50_trend']  = (df['Close'] > df['EMA50']).astype(int)

    df['trend_alignment'] = (
        (df['Close'] > df['EMA50']).astype(int) +
        (df['Close'] > df['EMA200']).astype(int) +
        (df['EMA50'] > df['EMA200']).astype(int)
          )  

    atr20_mean = df['ATR'].rolling(20).mean()

    df['pct_above_ema200_100'] = _pct_above(df['Close'], df['EMA200'], 100)
    df['r2_72']                = _rolling_r2(df['Close'], 72)
    df['ema50_200_atr']        = (df['EMA50'] - df['EMA200']) / atr20_mean
    df['donch_pos_100']        = _donch_pos(df['Close'], df['High'], df['Low'], 100)
    

    df.dropna(inplace=True)
    if df.empty:
        return pd.Series(index=expected_cols)
        

        
    
    features = pd.Series({
        'ema_diff':             df['ema_diff'].iloc[-1],
        'ema50_diff':           df['ema50_diff'].iloc[-1],
        'bb_penetration':       df['bb_penetration'].iloc[-1],
        'bb_lower_penetration': df['bb_lower_penetration'].iloc[-1],
        'bb_width':             df['BB_width'].iloc[-1],
        'atr_ratio':            df['atr_ratio'].iloc[-1],
        'adx':                  df['ADX'].iloc[-1],
        'macd_hist':            df['macd_hist'].iloc[-1],
        'ema200_slope':         df['ema200_slope'].iloc[-1],
        'vol_ratio':            df['vol_ratio'].iloc[-1],
        'trend_purity':         df['trend_purity'].iloc[-1],
        'EMA50_slope':          df['EMA50_slope'].iloc[-1],
        'vol_regime':           df['vol_regime'].iloc[-1],
        'rsi14':                df['RSI14'].iloc[-1],         
        'momentum_flag':        df['momentum_flag'].iloc[-1],
        'volume_ratio':         df['volume_ratio'].iloc[-1],
        'trend_volatility':     df['trend_volatility'].iloc[-1],
        'vwap':                 df['vwap'].iloc[-1],
        'vwap_distance':        df['vwap_distance'].iloc[-1],
        'CMO14':                df['CMO14'].iloc[-1],
        'ema_triple_diff':             df['ema_triple_diff'].iloc[-1],
        'EMA200_trend':         df['EMA200_trend'].iloc[-1],
        'EMA50_trend':          df['EMA50_trend'].iloc[-1],
        'trend_alignment':      df['trend_alignment'].iloc[-1],
        'pct_above_ema200_100': df['pct_above_ema200_100'].iloc[-1],
        'r2_72':                df['r2_72'].iloc[-1],
        'ema50_200_atr':        df['ema50_200_atr'].iloc[-1],
        'donch_pos_100':        df['donch_pos_100'].iloc[-1],
 
    }, index=expected_cols)
    return features


In [26]:
# ------------------ RISK MANAGEMENT ------------------
class RiskManager:
    def __init__(self):
        self.daily_start_equity = mt5.account_info().equity
        self.current_day = datetime.now().date()
        self.max_daily_loss = self.daily_start_equity * (1 - StrategyConfig.MAX_DD)
    
    def update_daily_values(self):
        now = datetime.now().date()
        if now != self.current_day:
            self.daily_start_equity = mt5.account_info().equity
            self.current_day = now
            self.max_daily_loss = self.daily_start_equity * (1 - StrategyConfig.MAX_DD)
    
    def check_daily_drawdown(self):
        current_equity = mt5.account_info().equity
        return current_equity < self.max_daily_loss


In [27]:
df_test = get_timeframe_data('EURUSD', '4H')
test_features = calculate_features(df_test, '4H')
print("Calculated features:", test_features)

Calculated features: ema_diff                 0.005169
ema50_diff               0.001260
bb_penetration          -0.001392
bb_lower_penetration     0.003722
bb_width                 0.005102
atr_ratio                0.972722
adx                     13.013763
macd_hist                0.000002
ema200_slope             0.000489
vol_ratio                1.721560
trend_purity            -0.003544
EMA50_slope              0.000348
vol_regime               0.000000
rsi14                   54.659183
momentum_flag           -0.000000
volume_ratio             1.326336
trend_volatility         0.176723
vwap                     1.171583
vwap_distance            0.000936
ema_triple_diff         -0.002648
CMO14                    0.237843
pct_above_ema200_100     0.840000
r2_72                    0.315161
ema50_200_atr            1.438992
donch_pos_100            0.742108
EMA200_trend             1.000000
trend_alignment          3.000000
dtype: float64


In [28]:
# ------------------ ENHANCED ORDER EXECUTION LOGIC ------------------

def calculate_position_size(self, symbol, risk_pct, timeframe):
    try:
        symbol_info = mt5.symbol_info(symbol)
        if not symbol_info:
            print(f"🚨 Failed to get {symbol} info")
            return None

        df = get_timeframe_data(symbol, timeframe)
        if df is None or df.empty:
            print(f"🚨 No data received for {symbol} on {timeframe} timeframe")
            return None

        atr = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'], 14).iloc[-1]
        if np.isnan(atr) or atr <= 0:
            print(f"⚠️ ATR value is invalid for {symbol}: {atr}")
            return None

        # ←—— ONLY THIS BLOCK CHANGED:
        key    = f"{symbol}.{timeframe}"
        config = StrategyConfig.INSTRUMENTS.get(key)
        if not config:
            print(f"🚨 No configuration found for {key}")
            return None

        equity      = mt5.account_info().equity
        risk_amount = equity * risk_pct
        print(f"Risk amount: ${risk_amount:.2f}")

        # use 1×ATR for SL on daily, otherwise 2×ATR
        sl_mult = 1 if timeframe == '1D' else 2
        sl_pips = (sl_mult * atr) / config['pip']
        if sl_pips <= 0 or risk_amount <= 0:
            print(f"⚠️ Invalid risk calculation: SL_pips={sl_pips}, Risk={risk_amount}")
            return None

        lot_size = risk_amount / (sl_pips * config.get('pip_value', 1))
        lot_size = np.clip(lot_size, symbol_info.volume_min, symbol_info.volume_max)
        lot_size = round(lot_size / symbol_info.volume_step) * symbol_info.volume_step

        print(f"🔢 {symbol} Lot Size on {timeframe}: {lot_size:.2f}")
        print(f"Lot Size Validation: Min={symbol_info.volume_min}, "
              f"Step={symbol_info.volume_step}, Calculated={lot_size:.2f}")
        return lot_size

    except Exception as e:
        print(f"🚨 Position size error for {symbol}: {str(e)}")
        return None


def validate_market_conditions(self, symbol):
    df = get_timeframe_data(symbol, '1H')
    if df is None or df.empty:
        return True
    if len(df) > 20:
        recent_vol = df['Close'].pct_change().std()
        threshold = 0.01
        if recent_vol > threshold:
            print(f"⚠️ High volatility detected for {symbol} "
                  f"(vol: {recent_vol:.5f} > {threshold}) - skipping trade")
            return False
    return True


def add_risk_parameters(self, symbol, signal, execution_price, request):
    df = get_timeframe_data(symbol, signal['timeframe'])
    if df is None or df.empty:
        print(f"⚠️ Not enough data for risk parameters on {symbol} for {signal['timeframe']} timeframe")
        return False
    atr = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'], 14).iloc[-1]
    if np.isnan(atr) or atr <= 0:
        print(f"⚠️ ATR is invalid for risk parameters on {symbol}: {atr}")
        return False
    if signal['direction'] == 'BUY':
        sl = execution_price - 2 * atr
        tp = execution_price + 3 * atr
    else:
        sl = execution_price + 2 * atr
        tp = execution_price - 3 * atr
    request['sl'] = sl
    request['tp'] = tp
    return True

def get_execution_price(symbol, direction):
    tick = mt5.symbol_info_tick(symbol)
    if not tick:
        print(f"🚨 Unable to retrieve tick data for {symbol}")
        return None
    return tick.ask if direction == 'BUY' else tick.bid

In [29]:
df = get_timeframe_data('EURUSD', '1H')
print("MT5 data columns:", df.columns.tolist())

MT5 data columns: ['Open', 'High', 'Low', 'Close', 'tick_volume', 'spread', 'real_volume']


In [3]:
# ------------------ AI TRADING ENGINE ------------------
class AITrader:
    def __init__(self, risk_manager):
        self.models = {}
        self.feature_orders = {}
        self.last_candle_times = {}
        

        # Load per‑timeframe models & expected feature order
        for sym_tf, cfg in StrategyConfig.INSTRUMENTS.items():
            try:
                pipe = joblib.load(cfg['model_path'])
                self.models[sym_tf] = pipe
                self.feature_orders[sym_tf] = pipe.named_steps['preprocessor'].feature_names_in_
            except Exception as e:
                print(f"🚨 Critical error loading {sym_tf} model: {e}")
                sys.exit(1)

        self.risk_manager       = risk_manager
        self.open_positions     = load_open_positions()
        self.tp_hit_flags       = {}
        self.trend_flag         = {}

    def has_existing_position(self, symbol, timeframe):
        """
        Return True if we already have an open position for this symbol+timeframe.
        """
        return any(
            pos for pos in self.open_positions
            if pos['symbol'] == symbol and pos.get('timeframe') == timeframe
        )

    def get_signal(self, symbol, timeframe):
        now = datetime.now()
        last_tp = self.tp_hit_flags.get(symbol)
        
 
        # freeze if TP hit today at or after 22:00
        if last_tp and last_tp.date() == now.date() and last_tp.time() >= dt_time(22, 0):
            print(f"⚠️ {symbol} hit TP at {last_tp.time():%H:%M} (≥ 22:00), blocking until next day.")
            return None

        try:
            full_key = f"{symbol}.{timeframe}"
            model    = self.models.get(full_key)
            if not model:
                return None
                
            # grab this instrument's threshold (or use the default)
            config    = StrategyConfig.INSTRUMENTS[full_key]
            threshold = config.get('threshold', StrategyConfig.DEFAULT_THRESHOLD)    

            df = get_timeframe_data(symbol, timeframe)
            if df is None or len(df) < 100:
                return None
            latest_bar = df.index[-1] 
            last_seen  = self.last_candle_times.get(full_key)
            if last_seen is not None and latest_bar <= last_seen:
                return None
            self.last_candle_times[full_key] = latest_bar   
               


            # —— calculate features & model predict —— 
            features = calculate_features(df, timeframe)
            cols        = self.feature_orders[full_key]
            features_df = features.to_frame().T[cols]
            missing = [f for f in cols if f not in features]
            if missing:
                print(f"🚨 Missing features for {symbol} on {timeframe}: {missing}")
                return None

            

            probs      = model.predict_proba(features_df)[0]
            confidence = max(probs)
            direction  = {1: 'BUY', 0: 'SELL'}.get(np.argmax(probs))

            # ── use the instrument-specific threshold ──
            print(
                f"{symbol} ({timeframe}) Prediction: "
                f"Probabilities={probs}, "
                f"Chosen Class={np.argmax(probs)}, "
                f"Confidence={confidence:.2f} | Threshold: {threshold}"
            )
            if confidence < threshold:
                print(f"✂️ Skipping {symbol} ({confidence:.2f} < {threshold:.2f})")
                return None

            signal = {
                'direction':  direction,
                'timeframe':  timeframe,
                'confidence': confidence
            }

              # BLOCK CONFLICTING TRADES BETWEEN MAJOR INDICES
            index_group = {'US30.cash', 'US100.cash', 'US500.cash'}
            if symbol in index_group:
                existing_trades = [t for t in self.open_positions if t['symbol'] in index_group]
                for t in existing_trades:
                    if t['trade_type'] != signal['direction']:
                        print(f"⛔ Index Conflict: Existing {t['trade_type']} on {t['symbol']} blocks new {signal['direction']} on {symbol}")
                        return None

           
            return signal  # Only return after all checks pass

        except Exception as e:
            print(f"🚨 Signal error for {symbol}.{timeframe}: {e}")
            return None
            
    def execute_trade(self, symbol, signal):
        try:
            # Conflict Detection & Reversal Resolution
            for trade in self.open_positions:
                if trade['symbol'] == symbol and trade['trade_type'] != signal['direction']:
                    existing_confidence = trade.get('confidence', 0.0)
                    if signal['confidence'] > existing_confidence + 0.05:
                        print(f"🔁 Reversal detected on {symbol}: Existing {trade['trade_type']} trade "
                              f"(conf {existing_confidence:.2f}, timeframe {trade['timeframe']}) will be closed "
                              f"for new {signal['direction']} signal (conf {signal['confidence']:.2f}, "
                              f"timeframe {signal['timeframe']}).")
                        positions = mt5.positions_get(ticket=trade['ticket'])
                        if positions and self.close_position(positions[0], reason="Reversal: Higher confidence new signal"):
                            self.open_positions.remove(trade)
                            save_open_positions(self.open_positions)
                        else:
                            print(f"⚠️ Reversal close order for {symbol} failed; trade remains open.")
                        break
                    else:
                        print(f"❌ Conflict on {symbol}: New signal confidence ({signal['confidence']:.2f}) "
                              f"is not high enough compared to existing trade ({existing_confidence:.2f}). Skipping trade.")
                        return

            # Skip if we already have a position on this timeframe
            if self.has_existing_position(symbol, signal['timeframe']):
                print(f"Trade for {symbol} on {signal['timeframe']} already exists. Skipping duplicate signal.")
                return

            # Basic checks
            if signal['direction'] not in ['BUY', 'SELL'] or self.risk_manager.check_daily_drawdown() \
               or len(self.open_positions) >= StrategyConfig.MAX_TRADES \
               or not verify_symbol(symbol) \
               or not self.validate_market_conditions(symbol):
                return

            # Get market data
            tick = mt5.symbol_info_tick(symbol)
            if not tick:
                print(f"🚨 Failed to get tick data for {symbol}")
                return

            # Position sizing
            risk_pct = StrategyConfig.RISK_PER_TRADE
            lot_size = self.calculate_position_size(symbol, risk_pct, signal['timeframe'])
            if lot_size is None:
                return

            # Execution price and spread
            execution_price = get_execution_price(symbol, signal['direction'])
            spread          = tick.ask - tick.bid

            # ↓↓↓ PATCH START: lookup by symbol.timeframe ↓↓↓
            full_key = f"{symbol}.{signal['timeframe']}"
            config   = StrategyConfig.INSTRUMENTS.get(full_key)
            if not config:
                print(f"🚨 No configuration found for {full_key}")
                return
            # ↑↑↑ PATCH END ↑↑↑

            # Determine multiplier
            if symbol == "XAUUSD":
                multiplier = 100
            elif symbol == "XAGUSD":
                multiplier = 50
            else:
                multiplier = 35

            # Suspicious‐price check using real pip
            if execution_price <= 0 or abs(execution_price - tick.ask) > (multiplier * config['pip']):
                print(f"⚠️ Suspicious price: {execution_price} vs tick {tick.ask}")
                return

            # Build trade request
            request = {
                "action":      mt5.TRADE_ACTION_DEAL,
                "symbol":      symbol,
                "volume":      lot_size,
                "type":        mt5.ORDER_TYPE_BUY if signal['direction']=="BUY" else mt5.ORDER_TYPE_SELL,
                "price":       execution_price,
                "deviation":   int(spread / config['pip'] + 2),
                "magic":       1001,
                "comment":     "PureStrategy_v8_Trade",
                "type_time":   mt5.ORDER_TIME_GTC,
                "type_filling":mt5.ORDER_FILLING_IOC
            }

            # Attach SL/TP
            if not add_risk_parameters(self, symbol, signal, execution_price, request):
                return

            # Send order with retries
            result = None
            for attempt in range(3):
                result = mt5.order_send(request)
                if result and result.retcode != mt5.TRADE_RETCODE_REQUOTE:
                    break
                if result and result.retcode == mt5.TRADE_RETCODE_REQUOTE:
                    tick = mt5.symbol_info_tick(symbol)
                    request['price'] = tick.ask if signal['direction']=="BUY" else tick.bid
                    print(f"⚠️ Requote occurred.Updating price to {request['price']:.5f} and retrying attempt {attempt+1}")
                time.sleep(1)

            if result is None:
                print("🚨 All order attempts failed")
                return

            # Handle broker response
            if hasattr(result, 'retcode'):
                print(f"Order Result: Retcode={result.retcode}, Comment='{result.comment}'")
                if result.retcode == mt5.TRADE_RETCODE_DONE:
                    print(f"✅ {signal['direction']} {symbol} executed on {signal['timeframe']}")
                    features  = calculate_features(get_timeframe_data(symbol, signal['timeframe']), signal['timeframe']).to_dict()
                    positions = mt5.positions_get(symbol=symbol)
                    if positions:
                        latest = max(positions, key=lambda p: p.time)
                        self.open_positions.append({
                            'symbol':        symbol,
                            'timeframe':     signal['timeframe'],
                            'ticket':        latest.ticket,
                            'entry_time':    datetime.now(),
                            'partial_closed': False,
                            'trade_type':    signal['direction'],
                            'entry_price':   execution_price,
                            'tp':            request['tp'],
                            'lot_size':      lot_size,
                            'features':      features,
                            'confidence':    signal['confidence']
                        })
                        save_open_positions(self.open_positions)
                    else:
                        print("⚠️ Could not retrieve open position after executing the trade.")
                else:
                    print(f"⚠️ Trade failed: {result.comment}")
            else:
                print("🚨 No valid response from broker!")

        except Exception as e:
            print(f"🔥 Critical execution error for {symbol}: {e}")

    def close_position(self, position, reason, volume=None):
        volume = volume if volume is not None else position.volume
        symbol = position.symbol
        # Ensure the symbol is selected
        if not mt5.symbol_select(symbol, True):
            print(f"⚠️ Failed to select symbol {symbol} for closing position.")
            return False

        # Determine close order details based on trade type
        if position.type == mt5.POSITION_TYPE_BUY:
            close_type = mt5.ORDER_TYPE_SELL
            tick = mt5.symbol_info_tick(symbol)
            price = tick.bid
        else:
            close_type = mt5.ORDER_TYPE_BUY
            tick = mt5.symbol_info_tick(symbol)
            price = tick.ask

        # Adjusted deviation parameter to allow more flexibility (increased from 20 to 30)
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "position": position.ticket,
            "symbol": symbol,
            "volume": volume,
            "type": close_type,
            "price": price,
            "deviation": 30,
            "magic": 1001,
            "comment": reason,
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_IOC,
        }
        for attempt in range(3):
            print(f"Attempt {attempt+1} to close position {symbol} with request: {request}")
            result = mt5.order_send(request)
            if result and hasattr(result, 'retcode'):
                print(f"Close order response for {symbol}: retcode = {result.retcode}, comment = '{result.comment}'")
                if result.retcode == mt5.TRADE_RETCODE_DONE:
                    print(f"🔒 Closed position {symbol} ({reason}) on attempt {attempt+1}")
                    self.trend_flag[symbol] = datetime.now()
                    return True
                elif result.retcode == mt5.TRADE_RETCODE_REQUOTE:
                    tick = mt5.symbol_info_tick(symbol)
                    if close_type == mt5.ORDER_TYPE_BUY:
                        request['price'] = tick.ask
                    else:
                        request['price'] = tick.bid
                    print(f"⚠️ Requote on closing {symbol} (attempt {attempt+1}), updating price to {request['price']:.5f} and retrying...")
            else:
                print(f"⚠️ No valid response for closing {symbol} on attempt {attempt+1}")
            time.sleep(1)
        print(f"⚠️ Failed to close position {symbol} after retries.")
        return False

    def calculate_position_size(self, symbol, risk_pct, timeframe):
        return calculate_position_size(self, symbol, risk_pct, timeframe)

    def validate_market_conditions(self, symbol):
        return validate_market_conditions(self, symbol)

    def add_risk_parameters(self, symbol, signal, execution_price, request):
        return add_risk_parameters(self, symbol, signal, execution_price, request)

    def manage_open_trades(self):
        for trade in list(self.open_positions):
            symbol = trade["symbol"]
            trade_type = trade["trade_type"]
            entry_price = trade["entry_price"]
            tp = trade["tp"]
            lot_size = trade["lot_size"]
            ticket = trade["ticket"]
            if trade.get("partial_closed", False):
                continue
            positions = mt5.positions_get(ticket=ticket)
            if not positions:
                print(f"Position {ticket} not found, removing from tracking")
                self.open_positions.remove(trade)
                save_open_positions(self.open_positions)
                continue
            position = positions[0]
            current_price = mt5.symbol_info_tick(symbol).ask if trade_type == "BUY" else mt5.symbol_info_tick(symbol).bid
            if trade_type == "BUY":
                required_price = entry_price + 0.7 * (tp - entry_price)
            else:
                required_price = entry_price - 0.7 * (entry_price - tp)
            if (trade_type == "BUY" and current_price >= required_price) or (trade_type == "SELL" and current_price <= required_price):
                partial_volume = round(lot_size * 0.70, 2)
                self.close_position(position, "Partial Close", partial_volume)
                trade["lot_size"] = round(lot_size - partial_volume, 2)
                trade["partial_closed"] = True
                self.tp_hit_flags[symbol] = datetime.now()
                symbol_info = mt5.symbol_info(symbol)
                spread = symbol_info.ask - symbol_info.bid
                new_sl = entry_price - spread if trade_type == "BUY" else entry_price + spread
                trade["sl"] = new_sl
                modify_request = {
                    "action": mt5.TRADE_ACTION_SLTP,
                    "position": ticket,
                    "symbol": symbol,
                    "sl": new_sl,
                    "tp": tp,
                    "magic": 1001
                }
                result = mt5.order_send(modify_request)
                if result and hasattr(result, 'retcode') and result.retcode == mt5.TRADE_RETCODE_DONE:
                    print(f"✅ Partial Close (70%) for {symbol}, SL adjusted to {new_sl:.2f}")
                else:
                    print(f"⚠️ Failed to adjust SL for {symbol}: {result.comment if result and hasattr(result, 'comment') else 'No valid result'}")
                save_open_positions(self.open_positions)

    def manage_positions(self):
       for trade in list(self.open_positions):
        positions = mt5.positions_get(symbol=trade['symbol'])
        if not positions:
            # Look back in history to see if it was closed by TP/SL
            utc_from = trade['entry_time'] - timedelta(hours=1)
            utc_to   = datetime.now()
            deals    = mt5.history_deals_get(utc_from, utc_to, group=trade['symbol'])

            # Match the closing deal by ticket
            close_deal = next((d for d in deals if d.ticket == trade['ticket']), None)

            if close_deal:
                reason = 'TP' if close_deal.profit > 0 else 'SL'
                print(f"⚠️ {trade['symbol']} closed by {reason} (profit={close_deal.profit:.2f})")
                if reason == 'TP':
                    self.tp_hit_flags[trade['symbol']] = datetime.now()
            else:
                print(f"⚠️ {trade['symbol']} closed manually or unknown reason.")

            print(f"Trade for {trade['symbol']} not found (likely manually closed).")
            self.trend_flag[trade['symbol']] = datetime.now()
            self.open_positions.remove(trade)
            save_open_positions(self.open_positions)
            continue



In [31]:
# ------------------ SAMPLE MARKET DATA FETCH ------------------
mt5.initialize()
symbols = ["US30.cash", "EURUSD", "XAUUSD"]

for symbol in symbols:
    print(f"\n📡 Fetching market data for {symbol}...")
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M1, 0, 10)
    if rates is None or len(rates) == 0:
        print(f"❌ No market data received for {symbol}. Possible reasons:")
        print("   - Market might be closed.")
        print("   - Broker might require a live account for full data access.")
        print("   - Symbol might have a different name.")
    else:
        df = pd.DataFrame(rates)
        df['time'] = pd.to_datetime(df['time'], unit='s')
        print(f"✅ Market data for {symbol} (Last 5 candles):")
        print(df.tail())


📡 Fetching market data for US30.cash...
✅ Market data for US30.cash (Last 5 candles):
                 time      open      high       low     close  tick_volume  \
5 2025-09-12 23:45:00  45850.78  45852.78  45850.28  45851.78            7   
6 2025-09-12 23:46:00  45852.28  45852.78  45851.28  45851.28            8   
7 2025-09-12 23:47:00  45850.78  45851.28  45847.18  45848.18            9   
8 2025-09-12 23:48:00  45847.18  45850.18  45847.18  45850.18           15   
9 2025-09-12 23:49:00  45848.68  45849.18  45843.18  45843.18           12   

   spread  real_volume  
5     243            0  
6     253            0  
7     253            0  
8     263            0  
9     263            0  

📡 Fetching market data for EURUSD...
✅ Market data for EURUSD (Last 5 candles):
                 time     open     high      low    close  tick_volume  \
5 2025-09-12 23:50:00  1.17294  1.17305  1.17289  1.17305           24   
6 2025-09-12 23:51:00  1.17304  1.17306  1.17303  1.17304        

In [ ]:
# ------------------ MAIN EXECUTION LOOP WITH CONNECTION RESILIENCE ------------------
def main_loop():
    reconnect_attempts = 0
    risk_manager = RiskManager()
    trader = AITrader(risk_manager)
    global TRADER

    

    logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
    logging.info("🟢 AI Trading System Active (v12)")
    logging.info(
        f"Risk: {StrategyConfig.RISK_PER_TRADE*100:.2f}% per trade, "
        f"Max Daily Drawdown: {StrategyConfig.MAX_DD*100:.2f}%, "
        f"Max Concurrent Trades: {StrategyConfig.MAX_TRADES}"
    )

    while True:
        loop_start = time.time()
        try:
            # only reconnect when actually disconnected
            if not mt5.terminal_info().connected:
                reconnect_attempts += 1
                print(f"🔄 Reconnecting (Attempt {reconnect_attempts}/5)...")
                initialize_mt5()
                time.sleep(10)
                continue
            # once connected, clear the counter
            reconnect_attempts = 0

            # update equity and check drawdown
            risk_manager.update_daily_values()
            if risk_manager.check_daily_drawdown():
                logging.warning("🛑 Daily drawdown limit reached – trading halted for today!")
                time.sleep(30)
                continue

            trader.manage_positions()
            trader.manage_open_trades()

            # ── delay scanning until 01:00 ───────────────────────────────
            now = datetime.now()
            day_start      = datetime.combine(now.date(), dt_time(0, 0))
            start_scanning = day_start + timedelta(hours=1, minutes=5)
            if now < start_scanning:
                logging.info(f"⏱ Waiting until {start_scanning.time()} to begin scanning")
                time.sleep((start_scanning - now).total_seconds())
                continue
                
            # ===== SIMPLE FRIDAY NIGHT BLOCK =====
            now = datetime.now()
            if (now.weekday() == 4 and now.time() >= dt_time(21, 0)) or now.weekday() in [5, 6]:
                logging.info("⏰ Weekend - Trading suspended until Monday 01:00")
                
                # Sleep until Monday 01:00
                days_ahead = (7 - now.weekday()) % 7  # days until Monday
                next_monday = now + timedelta(days=days_ahead)
                monday_start = datetime.combine(next_monday.date(), dt_time(1, 0))
                sleep_duration = (monday_start - now).total_seconds()
                time.sleep(sleep_duration)
                continue

            # Market scanning loop
            for sym_tf in StrategyConfig.INSTRUMENTS:
                symbol, timeframe = sym_tf.rsplit('.', 1)
                logging.info(f"Scanning {symbol} on {timeframe} timeframe...")
                if len(trader.open_positions) >= StrategyConfig.MAX_TRADES:
                    logging.warning("⚠️ Maximum concurrent trades reached!")
                    break

                signal = trader.get_signal(symbol, timeframe)
                if signal and signal['direction'] in ['BUY', 'SELL']:
                    logging.info(f"Signal for {symbol} on {timeframe}: {signal}")
                    trader.execute_trade(symbol, signal)

           

            # Trade status logging
            logging.info("----- Open Trades -----")
            for trade in trader.open_positions:
                logging.info(
                    f"Symbol: {trade['symbol']} | Timeframe: {trade.get('timeframe', 'unknown')} "
                    f"| Trade Type: {trade['trade_type']} | Entry Price: {trade['entry_price']}"
                )
            logging.info("-----------------------")


            elapsed = time.time() - loop_start
            sleep_time = max(0, 30 - elapsed)
            time.sleep(sleep_time)

        except Exception as e:
            print(f"Reconnecting due to error: {e}")
            mt5.shutdown()
            initialize_mt5()


# ------------------ MAIN EXECUTION ------------------
if __name__ == "__main__":
    while True:
        try:
            initialize_mt5()
            main_loop()
        except ConnectionError as e:
            print(f"🚨 Connection error: {str(e)}")
            if not reconnect_mt5():
                print("🛑 Permanent connection failure")
                break
        except KeyboardInterrupt:
            logging.info("🛑 Strategy stopped by user")
            break
        finally:
            mt5.shutdown()
            logging.info("✅ MT5 connection closed")

✅ Successfully connected to MT5 account: 550111923


2025-09-14 00:15:41,606 [INFO] 🟢 AI Trading System Active (v12)
2025-09-14 00:15:41,607 [INFO] Risk: 0.25% per trade, Max Daily Drawdown: 3.00%, Max Concurrent Trades: 7
2025-09-14 00:15:41,610 [INFO] ⏱ Waiting until 01:05:00 to begin scanning
